# AUV PPO Trajectory Training

This notebook is the single user-facing entry point for policy architecture, trajectory curriculum, simulation profile, Domain Randomization, reward policy, and training launch. Command construction lives in `simulation/isaac/workflows/common/trajectory_experiment.py`; the current project flow is summarized in the root `README.md`.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import importlib
import json
import os
import signal
import subprocess
import sys
import time

if Path(sys.prefix).name != "env_isaaclab":
    raise RuntimeError("请先执行 conda activate env_isaaclab，再启动本训练 notebook。")

REPO_ROOT = Path("/home/jining_yang/isaac-auv-env")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
# Reload the registry before workflow helpers. This makes newly added architecture
# profiles available in a running Jupyter kernel without a kernel restart.
importlib.invalidate_caches()
import simulation.isaac.agents.ppo.architectures.registry as architecture_registry
import simulation.isaac.agents.ppo.architectures as mlp_architectures
importlib.reload(architecture_registry)
importlib.reload(mlp_architectures)
import simulation.isaac.workflows.common.trajectory_experiment as experiment_tools
# Jupyter caches imported modules. Reload so this cell always uses the current dataclass/API definitions.
importlib.reload(experiment_tools)
from simulation.isaac.workflows.common.trajectory_experiment import (
    CompetenceGateCriteria, ExperimentSpec, TrainRequest, TrajectoryCurriculumRequest,
    benchmark_gpu_throughput, build_train_command, display_command,
    curriculum_segment_request, runs_dataframe, train_policy, launch_training_detached,
)
print("Experiment tools loaded")

# 唯一的网络结构入口。当前默认值使用 30-D 当前观测和 5 个历史样本的 MLP。
MLP_ARCHITECTURE = "mlp_history_5"
SPEC = ExperimentSpec(isaaclab_root=Path("/home/jining_yang/IsaacLab"), mlp_architecture=MLP_ARCHITECTURE)

# Each version owns its formula and coefficients in simulation/isaac/agents/rewards/policy_N.py.
# policy_0..5 retain earlier baselines. policy_6 uses Huber tracking residuals, applied-action
# regularization, and a true-termination cost; it does not double-count nose alignment.
REWARD_PROFILE = "policy_6"
TRAIN_SEED = 42
PPO_ROLLOUT_STEPS = 256
PPO_MAX_ITERATIONS = 500
# Actor 使用当前 30-D 可部署观测，并追加 5 个过去样本中的跟踪误差、姿态、角速度和实际动作；
# Critic 使用同一 135-D Actor 输入，再附加水流、有效水动力、刚体和推进器真值；
# 这些特权量只降低 PPO 价值估计方差，不会进入导出的真实机器人 Actor。
# 机器人状态只来自真实系统可提供的导航解算：
# 世界系位置/姿态、机体系线速度和角速度，由仿真状态直接形成策略观测。
# 目标状态及其导数由机载参考轨迹生成器精确计算，动作字段是控制器实际发送、经延迟/速率限制后的归一化命令。
# Actor 不接收水流、附加质量、阻尼、推进器内部真值或 DR 阶段；历史仅由控制器本身可以保存的观测构成。

# 轨迹课程的唯一人工配置入口。类型编号：0=椭圆/圆，1=Lissajous，2=单轴正弦，
# 3=波浪闭环（legacy helix），4=呼吸闭环（legacy spiral），5=Chirp，6=Racetrack，7=随机平滑 Fourier 曲线。
# 所有水平振幅均落在约 1.5 m 直径范围；重定时器在每条曲线局部施加临时运动学包络，
# 因此有效周期可能超过请求周期。日志记录实际有效周期和路径长度，不假设 40 s 恰好完成固定圈数。
# 训练轨迹固定为左右正弦、上下正弦和闭合空间螺旋线（type 8/9/10）。
# 每个环境在 reset 时独立均匀采样 0.1/0.2/0.3/0.4 m/s；正弦采用峰值速度，
# 空间螺旋线采用沿曲线恒定速度。所有轨迹保留 4 秒 C2 零速启动段。
TRAJECTORY_CURRICULUM = TrajectoryCurriculumRequest(
    enabled=True,
    amplitude_x_range=(0.60, 0.78),
    amplitude_y_range=(0.55, 0.75),
    amplitude_z_range=(0.08, 0.20),
    period_range=(10.0, 20.0),
    speed_levels_mps=(0.1, 0.2, 0.3, 0.4),
    stage_steps=(9_750, 22_500, 40_500),
    stage_0_types=(8, 9, 10),
    stage_1_types=(8, 9, 10),
    stage_2_types=(8, 9, 10),
    stage_3_types=(8, 9, 10),
    amplitude_scales=(0.55, 0.75, 0.90, 1.0),
    vertical_amplitude_scales=(0.25, 0.50, 0.75, 1.0),
    period_min_by_stage=(20.0, 10.0, 10.0, 10.0),
    period_max_by_stage=(20.0, 10.0, 10.0, 10.0),
)

# 额外水动力文件：当前没有实测水池数据，先使用建模预测。名义附加质量采用
# 561.500 x 401.999756 x 190.621773 mm 新 T60 外包络三轴椭球势流系数，并按独立排水体积 0.011304505834 m^3 修正；
# 并加入由对角项约束的 Y-r/Z-q 对称耦合，以及由现有二次阻尼和 CAD 投影面积估计的
# 高速攻角/侧滑项；它们均是可替换的工程先验，不是实测或 CFD。文件还包含周期水流、池壁和自由液面/晃荡参数，
# 并固定推进器命令速率为 4.0 归一化命令/s；训练和评测必须共用该文件，避免重新引入控制抖振。
# 后续拿到实测数据时只替换这个文件；不要在环境 config 中另设一套入口。
POOL_DYNAMICS_PROFILE = REPO_ROOT / "environment/hydrodynamics/coefficients/auv_pool_openfoam_hydrodynamics_v1.json"
# 课程文件：结构化 alpha/beta 高速均值力与随机 DR 是两个独立轴。前者随速度课程逐步启用，
# 后者仅覆盖残差不确定性；评估时应分别关闭两者，不能用更宽 DR 代替缺失均值力。
DOMAIN_RANDOMIZATION_SPEC = REPO_ROOT / "simulation/isaac/configs/domain_randomization/auv_pool_openfoam_hydrodynamics_dr_v1.json"

# curve_v2 is deliberately a fresh campaign namespace; policy_6 checkpoints are legacy evidence only.
RUN_NAME = f"curve_v2_{REWARD_PROFILE}"
RUN_TRAIN = True  # Set False to preview commands without launching anything.
# Default workflow: a detached supervisor runs train -> nominal eval -> robust eval -> gate.
# It survives VSCode/Jupyter closure and resumes from GATE_STATE_PATH when relaunched.
USE_COMPETENCE_GATE = True
GATE_SEGMENT_ITERATIONS = 25
GATE_RESTART = True  # True discards only the supervisor state; it never deletes checkpoints.
# Used only when USE_COMPETENCE_GATE=False.
TRAIN_DETACHED = True
# The selected MLP stores 135 actor inputs per transition. Benchmark before increasing
# the number of parallel environments or rollout steps.
GPU_ENV_CANDIDATES = [1024]

TRAIN = TrainRequest(
    reward_profile=REWARD_PROFILE,
    seed=TRAIN_SEED,
    num_envs=1024,
    run_name=RUN_NAME,
    headless=True,
    extra_args=("--logger", "tensorboard"),
    max_iterations=PPO_MAX_ITERATIONS,
    rollout_steps_per_env=PPO_ROLLOUT_STEPS,
    pool_dynamics_profile=POOL_DYNAMICS_PROFILE,
    domain_randomization_spec=DOMAIN_RANDOMIZATION_SPEC,
    trajectory_curriculum=TRAJECTORY_CURRICULUM,
)

print(f"Reward profile: {REWARD_PROFILE}")
print(f"Policy: {SPEC.architecture.name} ({SPEC.architecture.observation_dim}-D feed-forward Actor)")
print(f"Trajectory curriculum: {TRAJECTORY_CURRICULUM}")
print(display_command(build_train_command(SPEC, TRAIN), cwd=SPEC.isaaclab_root))

# The gate uses task metrics only: P95 position error, velocity RMSE, and boundary termination rate.
# Both sets must pass twice in a row. Keep these explicit here so each campaign is auditable.
GATE_CRITERIA = CompetenceGateCriteria()
GATE_STATE_PATH = SPEC.logs_root / "_curriculum" / f"{RUN_NAME}_supervisor_state.json"
GATE_CONFIG_PATH = SPEC.logs_root / "_curriculum" / f"{RUN_NAME}_campaign.json"
GATE_SUPERVISOR_SCRIPT = (
    "source/isaaclab_tasks/isaaclab_tasks/direct/isaac-auv-env/"
    "simulation/isaac/workflows/train/competence_curriculum.py"
)

def _gate_campaign_payload():
    train_payload = asdict(TRAIN)
    # JSON has no Path type; the supervisor reconstructs the same dataclasses.
    for name in ("pool_dynamics_profile", "domain_randomization_spec"):
        if train_payload[name] is not None:
            train_payload[name] = str(train_payload[name])
    return {
        "experiment": {
            "isaaclab_root": str(SPEC.isaaclab_root),
            "mlp_architecture": SPEC.mlp_architecture,
            "task_name": SPEC.task_name,
        },
        "train": train_payload,
        "criteria": asdict(GATE_CRITERIA),
        "segment_iterations": GATE_SEGMENT_ITERATIONS,
        "total_iterations": PPO_MAX_ITERATIONS,
        "state_path": str(GATE_STATE_PATH),
    }

GATE_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
GATE_CONFIG_PATH.write_text(json.dumps(_gate_campaign_payload(), indent=2) + "\n", encoding="utf-8")
GATE_COMMAND = ["./isaaclab.sh", "-p", GATE_SUPERVISOR_SCRIPT, "--config", str(GATE_CONFIG_PATH)]
if GATE_RESTART:
    GATE_COMMAND.append("--restart")
print("Competence gate state:", GATE_STATE_PATH)
print(display_command(GATE_COMMAND, cwd=SPEC.isaaclab_root))

## Existing runs for the selected architecture

In [ ]:
runs_df = runs_dataframe(SPEC, reward_profile=REWARD_PROFILE)
display(runs_df.head(20))

## Train

With `USE_COMPETENCE_GATE=True`, this launches a durable supervisor process. It trains 25 PPO iterations at the current gate stage, evaluates independent `curriculum_nominal` and full-DR `curriculum_robust` sets, records P95/RMSE/termination metrics, and only advances after two consecutive passes. The state file records the latest run and checkpoint for automatic recovery. Every real launch first stops an older process belonging to the same `RUN_NAME`, including a stranded Isaac Sim training child; it never touches a different experiment. Set it to `False` only to use the legacy single 500-iteration launch.

In [ ]:
if USE_COMPETENCE_GATE:
    print("[COMPETENCE GATE]")
    print(display_command(GATE_COMMAND, cwd=SPEC.isaaclab_root))
    def _supervisor_is_running(pid):
        # os.kill(pid, 0) is also true for a zombie process. A failed detached
        # supervisor must not block a restart or be reported as running.
        try:
            state = Path(f"/proc/{pid}/stat").read_text(encoding="utf-8").rsplit(") ", 1)[1][0]
        except (FileNotFoundError, IndexError):
            return False
        return state not in {"Z", "X"}

    def _command_line(pid):
        try:
            return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode("utf-8", "replace")
        except FileNotFoundError:
            return ""

    def _terminate_group(pid, label):
        # Each detached launcher starts its own session. Terminating the process
        # group therefore also closes Isaac Sim and its training subprocesses.
        try:
            group_id = os.getpgid(pid)
            os.killpg(group_id, signal.SIGTERM)
        except ProcessLookupError:
            return
        print(f"Stopped previous {label} process group (PID {pid}, PGID {group_id}).")
        deadline = time.monotonic() + 5.0
        while time.monotonic() < deadline and _supervisor_is_running(pid):
            time.sleep(0.1)
        if _supervisor_is_running(pid):
            try:
                os.killpg(group_id, signal.SIGKILL)
                print(f"Force-stopped previous {label} process group (PID {pid}).")
            except ProcessLookupError:
                pass

    def _stop_previous_campaign():
        candidate_pids = {}
        launcher_dir = SPEC.logs_root / "_launcher"
        # Verify cmdline as well as the pid file so a stale file cannot target
        # an unrelated PID that the operating system has later reused.
        for candidate in launcher_dir.glob(f"*_{RUN_NAME}_gate.pid"):
            try:
                candidate_pid = int(candidate.read_text(encoding="utf-8").strip())
            except ValueError:
                continue
            command = _command_line(candidate_pid)
            if _supervisor_is_running(candidate_pid) and "simulation/isaac/workflows/train/competence_curriculum.py" in command and RUN_NAME in command:
                candidate_pids[candidate_pid] = "competence supervisor"
        # A terminated supervisor may have left separately-sessioned train or
        # evaluation children. The state supplies the concrete run directory
        # used by the evaluator, so unrelated evaluations are not targeted.
        run_identifiers = {RUN_NAME}
        if GATE_STATE_PATH.is_file():
            try:
                latest_run = json.loads(GATE_STATE_PATH.read_text(encoding="utf-8")).get("latest_run")
                if latest_run:
                    run_identifiers.add(latest_run)
            except json.JSONDecodeError:
                pass
        process_listing = subprocess.run(["ps", "-eo", "pid=,args="], capture_output=True, text=True, check=False)
        run_marker = f"--run_name {RUN_NAME}"
        for process_line in process_listing.stdout.splitlines():
            is_training = "simulation/isaac/workflows/train/trajectory.py" in process_line and run_marker in process_line
            is_campaign_eval = "simulation/isaac/workflows/evaluate/trajectory.py" in process_line and any(
                f"--load_run {run_id}" in process_line for run_id in run_identifiers
            )
            if not (is_training or is_campaign_eval):
                continue
            try:
                candidate_pids[int(process_line.split(maxsplit=1)[0])] = "training" if is_training else "evaluation"
            except (IndexError, ValueError):
                continue
        for candidate_pid, label in candidate_pids.items():
            _terminate_group(candidate_pid, label)

    if RUN_TRAIN:
        _stop_previous_campaign()
    if RUN_TRAIN:
        launcher_dir = SPEC.logs_root / "_launcher"
        launcher_dir.mkdir(parents=True, exist_ok=True)
        launch_id = time.strftime("%Y-%m-%d_%H-%M-%S") + f"_{RUN_NAME}_gate"
        launcher_log = launcher_dir / f"{launch_id}.log"
        pid_path = launcher_dir / f"{launch_id}.pid"
        launch_env = os.environ.copy()
        launch_env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
        launch_env.setdefault("TERM", "xterm")
        launch_env.setdefault("PYTHONUNBUFFERED", "1")
        with launcher_log.open("w", encoding="utf-8") as log_file:
            log_file.write("[COMPETENCE CURRICULUM]\n" + " ".join(GATE_COMMAND) + "\n")
            gate_process = subprocess.Popen(
                GATE_COMMAND, cwd=SPEC.isaaclab_root, env=launch_env, stdin=subprocess.DEVNULL,
                stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True, close_fds=True,
            )
        pid_path.write_text(f"{gate_process.pid}\n", encoding="utf-8")
        print(f"Detached competence supervisor PID: {gate_process.pid}")
        print(f"Supervisor log: {launcher_log}")
        print(f"State: {GATE_STATE_PATH}")
        print(f"Stop only this campaign: kill -- -{gate_process.pid}")
elif TRAIN_DETACHED:
    train_pid, launcher_log = launch_training_detached(SPEC, TRAIN, execute=RUN_TRAIN)
    if train_pid:
        print(f"Training continues independently (PID {train_pid}).")
elif RUN_TRAIN:
    result, selected_run = train_policy(SPEC, TRAIN, execute=True)
    if selected_run:
        print(f"Completed run: {selected_run}")
else:
    train_policy(SPEC, TRAIN, execute=False)

## Training status snapshot

The supervisor is intentionally detached, so the launch cell returning is expected. Run the next cell again whenever a fresh snapshot is needed; it never blocks the notebook kernel. Use TensorBoard for scalar curves rather than repeatedly rendering the terminal log.

In [ ]:
import re

LOG_TAIL_LINES = 30

def _process_status(pid):
    try:
        state = Path(f"/proc/{pid}/stat").read_text(encoding="utf-8").rsplit(") ", 1)[1][0]
    except (FileNotFoundError, IndexError):
        return "exited"
    return "zombie" if state in {"Z", "X"} else "running"

def render_training_status():
    launcher_dir = SPEC.logs_root / "_launcher"
    pid_paths = sorted(launcher_dir.glob(f"*_{RUN_NAME}_gate.pid"), reverse=True)
    pid_path = pid_paths[0] if pid_paths else None
    pid = int(pid_path.read_text(encoding="utf-8").strip()) if pid_path else None
    status = _process_status(pid) if pid is not None else "not found"
    log_path = pid_path.with_suffix(".log") if pid_path else None
    print(f"Campaign: {RUN_NAME} | supervisor: {pid if pid is not None else 'not found'} ({status})")
    if GATE_STATE_PATH.is_file():
        state = json.loads(GATE_STATE_PATH.read_text(encoding="utf-8"))
        print(f"Gate: {state.get('status')} | completed: {state.get('completed_iterations', 0)}/{PPO_MAX_ITERATIONS} | stage: {state.get('stage')} | latest: {state.get('latest_checkpoint') or 'none'}")
    run_dirs = [path for path in SPEC.logs_root.glob(f"*_{RUN_NAME}") if path.is_dir()]
    if run_dirs:
        latest_run = max(run_dirs, key=lambda path: path.stat().st_mtime)
        checkpoints = sorted(latest_run.glob("model_*.pt"), key=lambda path: int(path.stem.split('_')[1]))
        print(f"Latest run: {latest_run.name} | checkpoint: {checkpoints[-1].name if checkpoints else 'none'}")
    else:
        print("Latest run: none")
    print(f"TensorBoard: tensorboard --logdir {SPEC.logs_root}")
    if log_path and log_path.is_file():
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-LOG_TAIL_LINES:]
        print(f"\n[log tail: {log_path.name}]")
        print(re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", "\n".join(tail)))
    elif log_path:
        print(f"Log has not been created yet: {log_path}")

render_training_status()

## Manual campaign stop and launcher cleanup

This cell is deliberately non-destructive by default. Set `CAMPAIGN_PROCESS_ACTION` to `stop` to terminate only processes belonging to the current `RUN_NAME`, or to `stop_and_clean` to additionally remove stale PID records. It never deletes checkpoints or logs.

In [ ]:
# Choose one: 'preview', 'stop', or 'stop_and_clean'.
CAMPAIGN_PROCESS_ACTION = 'stop_and_clean'
if CAMPAIGN_PROCESS_ACTION not in {'preview', 'stop', 'stop_and_clean'}:
    raise ValueError("CAMPAIGN_PROCESS_ACTION must be 'preview', 'stop', or 'stop_and_clean'.")

def _campaign_process_is_live(pid):
    try:
        state = Path(f'/proc/{pid}/stat').read_text(encoding='utf-8').rsplit(') ', 1)[1][0]
    except (FileNotFoundError, IndexError):
        return False
    return state not in {'Z', 'X'}

def _campaign_command_line(pid):
    try:
        return Path(f'/proc/{pid}/cmdline').read_bytes().replace(b'\0', b' ').decode('utf-8', 'replace')
    except FileNotFoundError:
        return ''

def _is_current_campaign_command(command, run_ids):
    is_train = 'simulation/isaac/workflows/train/trajectory.py' in command and f'--run_name {RUN_NAME}' in command
    is_supervisor = 'simulation/isaac/workflows/train/competence_curriculum.py' in command and RUN_NAME in command
    is_eval = 'simulation/isaac/workflows/evaluate/trajectory.py' in command and any(
        f'--load_run {run_id}' in command for run_id in run_ids
    )
    return is_train or is_supervisor or is_eval

def _campaign_process_group(pid):
    try:
        return os.getpgid(pid)
    except ProcessLookupError:
        return None

def _current_campaign_processes():
    run_ids = {RUN_NAME}
    if GATE_STATE_PATH.is_file():
        try:
            latest_run = json.loads(GATE_STATE_PATH.read_text(encoding='utf-8')).get('latest_run')
            if latest_run:
                run_ids.add(latest_run)
        except json.JSONDecodeError:
            pass

    launcher_dir = SPEC.logs_root / '_launcher'
    pid_records = {}
    for pattern in (f'*_{RUN_NAME}.pid', f'*_{RUN_NAME}_gate.pid'):
        for pid_path in launcher_dir.glob(pattern):
            try:
                pid_records[pid_path] = int(pid_path.read_text(encoding='utf-8').strip())
            except ValueError:
                continue

    processes = {}
    for pid_path, pid in pid_records.items():
        command = _campaign_command_line(pid)
        if _campaign_process_is_live(pid) and _is_current_campaign_command(command, run_ids):
            processes[pid] = (command, pid_path)

    listing = subprocess.run(['ps', '-eo', 'pid=,args='], capture_output=True, text=True, check=False)
    for line in listing.stdout.splitlines():
        try:
            pid_text, command = line.strip().split(maxsplit=1)
            pid = int(pid_text)
        except ValueError:
            continue
        if _is_current_campaign_command(command, run_ids):
            processes.setdefault(pid, (command, None))
    return processes, pid_records

processes, pid_records = _current_campaign_processes()
if processes:
    print(f'Current campaign processes for {RUN_NAME}:')
    for pid, (command, _) in sorted(processes.items()):
        pgid = _campaign_process_group(pid)
        print(f"  PID {pid} | PGID {pgid if pgid is not None else 'exited'} | {command}")
else:
    print(f'No running processes found for {RUN_NAME}.')

if CAMPAIGN_PROCESS_ACTION in {'stop', 'stop_and_clean'}:
    process_groups = {}
    for pid, (command, _) in processes.items():
        pgid = _campaign_process_group(pid)
        if pgid is None:
            continue
        if pgid == os.getpgrp():
            raise RuntimeError(f'Refusing to signal this notebook process group (PID {pid}, PGID {pgid}).')
        process_groups.setdefault(pgid, pid)
    for pgid, pid in sorted(process_groups.items()):
        try:
            os.killpg(pgid, signal.SIGTERM)
            print(f'Sent SIGTERM to campaign process group {pgid} (leader PID {pid}).')
        except ProcessLookupError:
            pass
    deadline = time.monotonic() + 8.0
    while time.monotonic() < deadline and any(_campaign_process_is_live(pid) for pid in processes):
        time.sleep(0.2)
    for pgid, pid in sorted(process_groups.items()):
        if _campaign_process_is_live(pid):
            try:
                os.killpg(pgid, signal.SIGKILL)
                print(f'Sent SIGKILL to remaining campaign process group {pgid} (leader PID {pid}).')
            except ProcessLookupError:
                pass

if CAMPAIGN_PROCESS_ACTION == 'stop_and_clean':
    removed = []
    for pid_path, pid in pid_records.items():
        if not _campaign_process_is_live(pid):
            pid_path.unlink(missing_ok=True)
            removed.append(pid_path.name)
    print('Removed stale PID records: ' + (', '.join(sorted(removed)) if removed else 'none'))
elif CAMPAIGN_PROCESS_ACTION == 'preview':
    print("Preview only. Set CAMPAIGN_PROCESS_ACTION = 'stop' or 'stop_and_clean' and re-run to act.")